In [0]:
%run "../lib/00_common"

In [0]:
%pip install polars

In [0]:
# ============================================================
# Gold: fact_orders
# ============================================================
import polars as pl
import os

# Leer todos los Silver necesarios
df_orders    = pl.read_parquet(f"{SILVER_ROOT}/orders/olist_orders_silver.parquet")
df_items     = pl.read_parquet(f"{SILVER_ROOT}/order_items/olist_order_items_silver.parquet")
df_payments  = pl.read_parquet(f"{SILVER_ROOT}/order_payments/olist_order_payments_silver.parquet")

print(f"orders:   {len(df_orders):,} filas")
print(f"items:    {len(df_items):,} filas")
print(f"payments: {len(df_payments):,} filas")

In [0]:
# ============================================================
# Gold: construir fact_orders
# ============================================================

# Base: order_items (granularidad a nivel de item)
fact_orders = df_items.join(
    df_orders.select([
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "delivery_days",
        "is_late",
    ]),
    on="order_id",
    how="left"
)

# Agregar total_paid_per_order desde payments
# Un registro por orden (evitar duplicados)
df_payments_agg = df_payments.select([
    "order_id",
    "total_paid_per_order"
]).unique(subset=["order_id"], keep="first")

fact_orders = fact_orders.join(
    df_payments_agg,
    on="order_id",
    how="left"
)

# Agregar fecha como Date para join con dim_date
fact_orders = fact_orders.with_columns([
    pl.col("order_purchase_timestamp").cast(pl.Date).alias("order_purchase_date")
]).drop("order_purchase_timestamp")

print(f" fact_orders construido.")
print(f"   Filas:    {len(fact_orders):,}")
print(f"   Columnas: {len(fact_orders.columns)}")
print(f"\nColumnas: {fact_orders.columns}")

In [0]:
# ============================================================
# Gold: guardar fact_orders
# ============================================================

gold_fact_orders_path = f"{GOLD_ROOT}/fact_orders/fact_orders.parquet"
os.makedirs(os.path.dirname(gold_fact_orders_path), exist_ok=True)
fact_orders.write_parquet(gold_fact_orders_path)

print(f"✅ fact_orders guardado.")
print(f"   Filas:    {len(fact_orders):,}")
print(f"   Columnas: {len(fact_orders.columns)}")
print(f"   Ruta:     {gold_fact_orders_path}")

In [0]:
# ============================================================
# Registrar tablas Gold en Unity Catalog como Delta
# ============================================================

tablas = {
    "fact_orders":   f"{GOLD_ROOT}/fact_orders/fact_orders.parquet",
    "dim_customers": f"{GOLD_ROOT}/dim_customers/dim_customers.parquet",
    "dim_sellers":   f"{GOLD_ROOT}/dim_sellers/dim_sellers.parquet",
    "dim_products":  f"{GOLD_ROOT}/dim_products/dim_products.parquet",
    "dim_payment":   f"{GOLD_ROOT}/dim_payment/dim_payment.parquet",
    "dim_date":      f"{GOLD_ROOT}/dim_date/dim_date.parquet",
}

for nombre, path in tablas.items():
    # Leer Parquet con Spark
    df_spark = spark.read.parquet(path)
    
    # Escribir como tabla Delta en Unity Catalog
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"workspace.elt_project.{nombre}")
    
    print(f" Tabla registrada: workspace.elt_project.{nombre}")

print("\n Todas las tablas registradas en Unity Catalog.")